# 03 – Modelling

Train and evaluate classifiers to predict stress / affect state from biosignal features.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_loader import load_dataset, BIOSIGNAL_FEATURE_COLS
from src.preprocessing import drop_missing, impute_missing, encode_phase, normalise_features
from src.features import select_features_variance
from src.model import (
    split_data,
    train_random_forest,
    train_logistic_regression,
    evaluate_classifier,
    cross_validate_model,
)

sns.set_theme(style='whitegrid')
%matplotlib inline

In [ ]:
df = load_dataset('../data/raw')
df = drop_missing(df)
df = impute_missing(df)
df = encode_phase(df)
print(df.shape)

In [ ]:
# Use phase_encoded as the target: classify pre (0) vs puzzle (1)
model_df = df[df['phase'].isin(['pre', 'puzzle'])].copy()
feature_cols = select_features_variance(model_df, BIOSIGNAL_FEATURE_COLS, threshold=0.01)
model_df = normalise_features(model_df, feature_cols)

X_train, X_test, y_train, y_test = split_data(model_df, 'phase_encoded', feature_cols)
print(f'Train: {len(X_train)}, Test: {len(X_test)}')

In [ ]:
# Random Forest
rf = train_random_forest(X_train, y_train, n_estimators=100)
rf_metrics = evaluate_classifier(rf, X_test, y_test)
print('Random Forest:', rf_metrics)

In [ ]:
# Logistic Regression
lr = train_logistic_regression(X_train, y_train)
lr_metrics = evaluate_classifier(lr, X_test, y_test)
print('Logistic Regression:', lr_metrics)

In [ ]:
# 5-fold cross-validation
from sklearn.ensemble import RandomForestClassifier
cv_clf = RandomForestClassifier(n_estimators=100, random_state=42)
X_all = model_df[feature_cols]
y_all = model_df['phase_encoded']
cv_result = cross_validate_model(cv_clf, X_all, y_all, cv=5)
print(f"CV F1 (weighted): {cv_result['mean']:.3f} ± {cv_result['std']:.3f}")

In [ ]:
# Feature importances
importances = pd.Series(rf.feature_importances_, index=feature_cols).sort_values(ascending=False)
importances.head(20).plot(kind='barh', figsize=(8, 6), title='Top 20 feature importances (RF)')
plt.xlabel('Importance')
plt.tight_layout()
plt.savefig('../results/figures/feature_importances.png', dpi=150)
plt.show()

In [ ]:
# Save metrics summary
summary = pd.DataFrame([rf_metrics, lr_metrics], index=['RandomForest', 'LogisticRegression'])
summary.to_csv('../results/tables/model_metrics.csv')
summary